https://github.com/tensorflow/text/blob/master/examples/keras_example_174.ipynb

In [2]:
import tensorflow as tf
import tensorflow_text as text
     

2026-04-04 22:03:23.928978: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775358203.943241  205017 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775358203.947378  205017 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-04 22:03:23.963204: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
ragged_input = tf.ragged.constant([[1, 2, 3, 4, 5], [5, 6]])
input_data = tf.data.Dataset.from_tensor_slices(ragged_input).batch(2)

model = tf.keras.Sequential([
  tf.keras.layers.InputLayer(input_shape=(None,), dtype='int32', ragged=True),
  text.keras.layers.ToDense(pad_value=0, mask=True),
  tf.keras.layers.Embedding(100, 16),
  tf.keras.layers.LSTM(32),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
  optimizer="rmsprop",
  loss="binary_crossentropy",
  metrics=["accuracy"])

output = model.predict(input_data)
print(output)

I0000 00:00:1775358205.404676  205017 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22462 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1775358205.405261  205017 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22462 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:c1:00.0, compute capability: 8.6
/home/tf/miniconda3/envs/tf-new/lib/python3.10/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
/home/tf/miniconda3/envs/tf-new/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'embedding' (of type Embedding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 766ms/step
[[0.49877545]
 [0.5003491 ]]


I0000 00:00:1775358206.974827  205392 cuda_dnn.cc:529] Loaded cuDNN version 90501


In [4]:
def _CreateTable(vocab, num_oov=1):
  init = tf.lookup.KeyValueTensorInitializer(
      vocab,
      tf.range(tf.size(vocab, out_type=tf.int64), dtype=tf.int64),
      key_dtype=tf.string,
      value_dtype=tf.int64)
  return tf.lookup.StaticVocabularyTable(
      init, num_oov, lookup_key_dtype=tf.string)

reviews_data_array = ['I really liked this movie', 'not my favorite']
reviews_labels_array = [1,0]
train_x = tf.constant(reviews_data_array)
train_y = tf.constant(reviews_labels_array)

a = _CreateTable(['I', 'really', 'liked', 'this', 'movie', 'not', 'my', 'favorite'])

def preprocess(data, labels):
  t = text.WhitespaceTokenizer()
  data = t.tokenize(data)
  # data = data.merge_dims(-2,-1)
  ids = tf.ragged.map_flat_values(a.lookup, data)
  return (ids, labels)

train_dataset = tf.data.Dataset.from_tensor_slices((train_x, train_y)).batch(2)
train_dataset = train_dataset.map(preprocess)

model = tf.keras.Sequential([
  tf.keras.layers.InputLayer(input_shape=(None,), dtype='int64', ragged=True),
  text.keras.layers.ToDense(pad_value=0, mask=True),
  tf.keras.layers.Embedding(100, 16),
  tf.keras.layers.LSTM(32),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
  optimizer="rmsprop",
  loss="binary_crossentropy",
  metrics=["accuracy"])

output = model.fit(train_dataset, epochs=1, verbose=1)
print(output)

/home/tf/miniconda3/envs/tf-new/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'embedding_1' (of type Embedding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5000 - loss: 0.6947
